# 01 Web Panel Launcher

This notebook can launch the web panel for you.

When you run the launch cell below, it is equivalent to running the server command in a terminal.

The web panel is the main beginner tool for:

- camera check
- motor check
- motor calibration
- mask tuning
- automatic line-following test


## Safety

Put the rover on a stand before testing motor movement. Do not put it on the floor until camera, serial, and calibration all look good.


In [ ]:
import glob
import os
import socket
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
video_devices = sorted(glob.glob('/dev/video*'))
serial_candidates = [p for p in ('/dev/ttyTHS1', '/dev/ttyTHS2') if os.path.exists(p)]
hostname = socket.gethostname()
ip_addresses = sorted({addr[-1][0] for addr in socket.getaddrinfo(hostname, None, family=socket.AF_INET)})

print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial candidates:', serial_candidates or ['none found'])
print('Host IPv4:', ip_addresses or ['127.0.0.1'])


In [ ]:
camera_source = 'usb'
host = '0.0.0.0'
http_port = 8765
serial_port = serial_candidates[0] if serial_candidates else '/dev/ttyTHS1'
baudrate = 115200
sensor_id = 0
device_index = 0
frame_width = 1280
frame_height = 720
warmup_frames = 12

print('Edit the values in this cell if needed, then run the next cells.')


In [ ]:
import shlex
import subprocess
import sys
import time
from IPython.display import HTML, display

if '_teleop_server_process' not in globals():
    _teleop_server_process = None

teleop_log_path = project_root / '.teleop_server.notebook.log'

def teleop_python() -> str:
    candidate = project_root / '.venv' / 'bin' / 'python'
    return str(candidate) if candidate.exists() else sys.executable

def teleop_command() -> list[str]:
    return [
        teleop_python(),
        str(project_root / 'scripts' / 'teleop_server.py'),
        '--host', host,
        '--http-port', str(http_port),
        '--camera-source', camera_source,
        '--sensor-id', str(sensor_id),
        '--device-index', str(device_index),
        '--width', str(frame_width),
        '--height', str(frame_height),
        '--warmup-frames', str(warmup_frames),
        '--port', serial_port,
        '--baudrate', str(baudrate),
    ]

def stop_teleop_server() -> None:
    global _teleop_server_process
    if _teleop_server_process is None:
        return
    if _teleop_server_process.poll() is None:
        _teleop_server_process.terminate()
        try:
            _teleop_server_process.wait(timeout=3)
        except subprocess.TimeoutExpired:
            _teleop_server_process.kill()
            _teleop_server_process.wait(timeout=3)
    _teleop_server_process = None

def browser_url() -> str:
    browser_host = '127.0.0.1' if host == '0.0.0.0' else host
    return f'http://{browser_host}:{http_port}'

print('Equivalent terminal command:')
print(shlex.join(teleop_command()))


In [ ]:
stop_teleop_server()
cmd = teleop_command()
print('Starting web panel...')
print(shlex.join(cmd))
with open(teleop_log_path, "w") as log_file:
    _teleop_server_process = subprocess.Popen(
        cmd,
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
time.sleep(2.0)
if _teleop_server_process.poll() is not None:
    raise RuntimeError(f'web panel exited early; check {teleop_log_path}')
print('Web panel running at:', browser_url())
if ip_addresses:
    print('LAN example:', f'http://{ip_addresses[0]}:{http_port}')
display(HTML(f"<p><a href=\"{browser_url()}\" target=\"_blank\">Open the web panel in a new tab</a></p><iframe src=\"{browser_url()}\" width=\"100%\" height=\"720\" style=\"border:1px solid #ccc; border-radius:12px;\"></iframe>"))


In [ ]:
print('Last web panel log lines:')
if teleop_log_path.exists():
    print('\n'.join(teleop_log_path.read_text().splitlines()[-40:]))
else:
    print('No log file yet.')


In [ ]:
#stop_teleop_server()
#print('Web panel stopped.')


## What To Do In The Panel

1. confirm `Camera` says ready
2. confirm `Serial` says ready
3. confirm the live RGB image updates
4. press `Forward`, `Back`, `Left`, and `Right` carefully
5. calibrate `forward`, `back`, `left`, and `right` with the calibration buttons
6. tune the mask if needed
7. press `Auto Start` for the line-following test
8. press `Stop Auto` or `Stop` if anything looks wrong
